# VIVO Backgammon — Entrenar en GPU (Kaggle)

Red wide-net (198→256→128→64→1, tanh), idéntica a la que usa el navegador VIVO (`nn-model.ts`).
Entrena por TD(0) contra el heurístico, con auto-stop cuando vence al heurístico.

## Pasos (en Kaggle):
1. Sube este .ipynb a Kaggle: **Notebooks → New Notebook → File → Import Notebook** (sube este archivo).
2. Arriba a la derecha, icono de **engranaje (Settings)** → **Accelerator = GPU** (T4 ×2 o P100) → Save.
3. **Run → Run all**.
4. Al terminar, la última celda te da un enlace para descargar `model_weights.json`.

Luego coloca ese archivo en `public/` de tu proyecto VIVO y recarga el navegador.

In [ ]:
# Celda 0 — descarga el motor y el entrenador del repo, y confirma la GPU
import urllib.request, os
base = 'https://raw.githubusercontent.com/Pirzl/Backgammon-AR-Pro/260816-gpu-train/colab/'
for f in ['bg_engine.py', 'bg_net.py']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(base + f, f)
        print('descargado', f)

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__)
print('GPU disponible:', gpus)
if not gpus:
    raise SystemExit('NO HAY GPU: activa el Accelerator=GPU en Settings (engranaje arriba a la derecha).')
print('OK: entrenamiento en GPU.')

In [ ]:
# Celda 1 — entrena (primera corrida: 20.000 partidas). Ajusta --games para más.
!python bg_net.py --games 20000 --opponent heuristic --label td0 --exploration 0.15 \
  --max-moves 400 --epochs 3 --eval-every 200 --eval-games 100 --save-every 50 \
  --stop-rate 0.55 --stop-streak 2 --out model_weights.json --seed 1

In [ ]:
# Celda 2 — verifica que la red NO haya colapsado (no salga todo igual / neuronas muertas)
import json, math, random
import bg_engine as bg

raw = json.load(open('model_weights.json'))
w = raw['weights']
W1, b1, W2, b2, W3, b3, W4, b4 = [l['data'] for l in w]

def mat(M, input_dim, x, b):
    return [b[j] + sum(x[i] * M[i * len(b) + j] for i in range(input_dim)) for j in range(len(b))]

def fwd(fv):
    h1 = [max(0, v) for v in mat(W1, 198, fv, b1)]
    h2 = [max(0, v) for v in mat(W2, 256, h1, b2)]
    h3 = [max(0, v) for v in mat(W3, 128, h2, b3)]
    y = [math.tanh(v) for v in mat(W4, 64, h3, b4)]
    return y[0], (h1, h2, h3)

rng = random.Random(1)
preds = []
hs = ([], [], [])
for t in ['white', 'black']:
    y, h = fwd(bg.encode_board(bg.INITIAL_BOARD, t))
    preds.append(y)
    for _ in range(8):
        b = [0] * 30
        for _ in range(15):
            b[rng.randint(1, 24)] += 1
        for _ in range(15):
            b[rng.randint(1, 24)] -= 1
        y, h = fwd(bg.encode_board(b, t))
        preds.append(y)
        for li, hh in enumerate(h):
            hs[li].append(hh)

mean = sum(preds) / len(preds)
std = math.sqrt(sum((p - mean) ** 2 for p in preds) / len(preds))
print('preds =', [round(p, 3) for p in preds], 'std = %.4f' % std)

dead = []
for st in hs:
    n_dead = sum(1 for u in range(len(st[0])) if all(abs(st[k][u]) < 1e-9 for k in range(len(st))))
    dead.append(n_dead / len(st[0]))
print('dead =', ['%.0f%%' % (x * 100) for x in dead])

if std < 1e-3:
    print('VEREDICTO: COLLAPSED')
elif any(x > 0.5 for x in dead):
    print('VEREDICTO: WARNING (unidades muertas)')
else:
    print('VEREDICTO: ALIVE')

In [ ]:
# Celda 3 — descarga el modelo entrenado
from IPython.display import FileLink
FileLink('model_weights.json')